In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/playground-series-s5e10/sample_submission.csv
/kaggle/input/playground-series-s5e10/train.csv
/kaggle/input/playground-series-s5e10/test.csv


In [2]:
import os

# List everything in /kaggle/input
for dirname, _, filenames in os.walk('/kaggle/input'):
    print(dirname)
    for filename in filenames:
        print("   ", filename)

/kaggle/input
/kaggle/input/playground-series-s5e10
    sample_submission.csv
    train.csv
    test.csv


In [3]:
import pandas as pd

# Load datasets
train = pd.read_csv("/kaggle/input/playground-series-s5e10/train.csv")
test = pd.read_csv("/kaggle/input/playground-series-s5e10/test.csv")
submission = pd.read_csv("/kaggle/input/playground-series-s5e10/sample_submission.csv")
#prints the number of rows in each Dataframe to see how big the dataset is 
#main dataset with input features and target column
print("Train shape:", train.shape)
#only the input features
print("Test shape:", test.shape)
#template showing you the format
print("Submission shape:", submission.shape)

train.head()

Train shape: (517754, 14)
Test shape: (172585, 13)
Submission shape: (172585, 2)


,id,road_type,num_lanes,curvature,speed_limit,lighting,weather,road_signs_present,public_road,time_of_day,holiday,school_season,num_reported_accidents,accident_risk
0,0,urban,2,0.06,35,daylight,rainy,False,True,afternoon,False,True,1,0.13
1,1,urban,4,0.99,35,daylight,clear,True,False,evening,True,True,0,0.35
2,2,rural,4,0.63,70,dim,clear,False,True,morning,True,False,2,0.30
3,3,highway,4,0.07,35,dim,rainy,True,True,morning,False,False,1,0.21
4,4,rural,1,0.58,60,daylight,foggy,False,False,evening,True,False,1,0.56


In [4]:
# Look at the first 5 rows
train.head()

# how many non-null values and datatypes(important for preprocessing)
train.info()

# Summary statistics for numeric columns
train.describe()

# Count missing values per column
train.isnull().sum().sort_values(ascending=False).head(10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 517754 entries, 0 to 517753
Data columns (total 14 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   id                      517754 non-null  int64  
 1   road_type               517754 non-null  object 
 2   num_lanes               517754 non-null  int64  
 3   curvature               517754 non-null  float64
 4   speed_limit             517754 non-null  int64  
 5   lighting                517754 non-null  object 
 6   weather                 517754 non-null  object 
 7   road_signs_present      517754 non-null  bool   
 8   public_road             517754 non-null  bool   
 9   time_of_day             517754 non-null  object 
 10  holiday                 517754 non-null  bool   
 11  school_season           517754 non-null  bool   
 12  num_reported_accidents  517754 non-null  int64  
 13  accident_risk           517754 non-null  float64
dtypes: bool(4), float64(

id                    0
road_type             0
num_lanes             0
curvature             0
speed_limit           0
lighting              0
weather               0
road_signs_present    0
public_road           0
time_of_day           0
dtype: int64

In [5]:
# check if it's a balanced dataset looking at the percentage of 1s
print("TARGET VARIABLE")
target_counts = train['accident_risk'].value_counts()
print("Accident risk values:")
print("  0:", target_counts[0])
print("  1:", target_counts[1])
print("Percentage of 1s:", (target_counts[1] / len(train)) * 100)


TARGET VARIABLE
Accident risk values:
  0: 608
  1: 61
Percentage of 1s: 0.011781656925875995


In [6]:
print("NUMERICAL COLUMNS BASIC STATS")
numerical_cols = ['num_lanes', 'curvature', 'speed_limit', 'num_reported_accidents']
for col in numerical_cols:
    print(f"{col}: min={train[col].min()}, max={train[col].max()}")

NUMERICAL COLUMNS BASIC STATS
num_lanes: min=1, max=4
curvature: min=0.0, max=1.0
speed_limit: min=25, max=70
num_reported_accidents: min=0, max=7


In [7]:
print("CATEGORICAL COLUMNS")
categorical_cols = ['road_type', 'lighting', 'weather', 'time_of_day']
for col in categorical_cols:
    unique_values = train[col].unique()
    print(f"{col}: {list(unique_values)}")


CATEGORICAL COLUMNS
road_type: ['urban', 'rural', 'highway']
lighting: ['daylight', 'dim', 'night']
weather: ['rainy', 'clear', 'foggy']
time_of_day: ['afternoon', 'evening', 'morning']


In [8]:
# Encode all categorical features simply
categorical_cols = ['road_type', 'lighting', 'weather', 'time_of_day']

for col in categorical_cols:
    # Get all unique values from both train and test
    all_values = pd.concat([train[col], test[col]]).unique()
    # Create mapping: category -> number
    mapping = {value: idx for idx, value in enumerate(all_values)}
    
    # Apply mapping
    train[f'{col}_encoded'] = train[col].map(mapping)
    test[f'{col}_encoded'] = test[col].map(mapping)
    
    print(f"{col} mapping: {mapping}")

# Convert boolean to int
bool_cols = ['road_signs_present', 'public_road', 'holiday', 'school_season']
for col in bool_cols:
    train[f'{col}_int'] = train[col].astype(int)
    test[f'{col}_int'] = test[col].astype(int)



road_type mapping: {'urban': 0, 'rural': 1, 'highway': 2}
lighting mapping: {'daylight': 0, 'dim': 1, 'night': 2}
weather mapping: {'rainy': 0, 'clear': 1, 'foggy': 2}
time_of_day mapping: {'afternoon': 0, 'evening': 1, 'morning': 2}


In [9]:
target_column = 'accident_risk'
final_train_data = train.drop(columns=[
    'id', target_column 
])


In [10]:
X = final_train_data
y = train[target_column] # The target variable remains the same

In [11]:
X.head()

,road_type,num_lanes,curvature,speed_limit,lighting,weather,road_signs_present,public_road,time_of_day,holiday,school_season,num_reported_accidents,road_type_encoded,lighting_encoded,weather_encoded,time_of_day_encoded,road_signs_present_int,public_road_int,holiday_int,school_season_int
0,urban,2,0.06,35,daylight,rainy,False,True,afternoon,False,True,1,0,0,0,0,0,1,0,1
1,urban,4,0.99,35,daylight,clear,True,False,evening,True,True,0,0,0,1,1,1,0,1,1
2,rural,4,0.63,70,dim,clear,False,True,morning,True,False,2,1,1,1,2,0,1,1,0
3,highway,4,0.07,35,dim,rainy,True,True,morning,False,False,1,2,1,0,2,1,1,0,0
4,rural,1,0.58,60,daylight,foggy,False,False,evening,True,False,1,1,0,2,1,0,0,1,0


In [12]:
from sklearn.utils import class_weight
import numpy as np

# Calculate class weights
class_weights = class_weight.compute_class_weight(
    'balanced',
    classes=np.unique(y),
    y=y
)
print(f"Class weights: {class_weights}")

# Or use scale_pos_weight for XGBoost
scale_pos_weight = len(y) / sum(y) - 1
print(f"XGBoost scale_pos_weight: {scale_pos_weight:.2f}")

Class weights: [  8.6894804    3.27539001   2.3111129    2.48621369   2.63896308
   1.80869705   1.13641731   1.47575533   1.23902535   1.32344792
   7.52593174   0.9339233    0.93840215   0.8130508    0.90729934
   0.7364377    0.72204511   0.59555902   0.61885956   0.53987371
   0.69534142   0.49752369   0.52283069   0.53849802   0.51398036
   0.48487556   0.50125276   0.47868117   0.39762204   0.46089192
   0.47306627   0.48469762   0.37101152   0.45829321   0.33993077
   0.42844896   0.44056071   0.39271568   0.46478438   0.42269014
   0.48536556   0.5196935    0.49954653   0.5196935    0.49398823
   0.53571325   0.66884467   0.56869796   0.63165998   0.65080119
   0.90003477   0.78748011   0.89002764   0.85254221   0.93375823
   1.19882099   0.99928203   1.21397153   1.24987085   1.427893
   1.36411156   1.45703367   1.5932461    1.85636124   1.7299293
   1.90660559   2.28908322   2.25681507   2.23580367   2.05093326
   3.94268961   3.0450744    3.54815586   3.47808037   4.6181853

In [13]:
# 1. Finish encoding ALL categorical columns
from sklearn.preprocessing import LabelEncoder

# Make copies to avoid modifying originals
train_encoded = train.copy()
test_encoded = test.copy()

# Encode ALL object columns
object_cols = ['road_type', 'lighting', 'weather', 'time_of_day']

for col in object_cols:
    le = LabelEncoder()
    # Combine train + test to handle all categories
    combined = pd.concat([train_encoded[col], test_encoded[col]])
    le.fit(combined)
    train_encoded[col] = le.transform(train_encoded[col])
    test_encoded[col] = le.transform(test_encoded[col])

# Convert boolean to int
bool_cols = ['road_signs_present', 'public_road', 'holiday', 'school_season']
for col in bool_cols:
    train_encoded[col] = train_encoded[col].astype(int)
    test_encoded[col] = test_encoded[col].astype(int)

print("✅ All encoding done!")

# 2. Prepare data for modeling
X = train_encoded.drop(['accident_risk', 'id'], axis=1)
y = train_encoded['accident_risk']
X_test = test_encoded.drop(['id'], axis=1)

print(f"X shape: {X.shape}, X_test shape: {X_test.shape}")

# 3. Check data types
print("Data types:")
print(X.dtypes)

# 4. Use simple regression model
from xgboost import XGBRegressor

model = XGBRegressor(random_state=42)
model.fit(X, y)

# 5. Make predictions
predictions = model.predict(X_test)

# 6. Create submission
submission['accident_risk'] = predictions
submission.to_csv('simple_submission.csv', index=False)
print("✅ Submission file created!")

# Check results
print(f"Predictions range: {predictions.min():.3f} to {predictions.max():.3f}")

✅ All encoding done!
X shape: (517754, 20), X_test shape: (172585, 20)
Data types:
road_type                   int64
num_lanes                   int64
curvature                 float64
speed_limit                 int64
lighting                    int64
weather                     int64
road_signs_present          int64
public_road                 int64
time_of_day                 int64
holiday                     int64
school_season               int64
num_reported_accidents      int64
road_type_encoded           int64
lighting_encoded            int64
weather_encoded             int64
time_of_day_encoded         int64
road_signs_present_int      int64
public_road_int             int64
holiday_int                 int64
school_season_int           int64
dtype: object
✅ Submission file created!
Predictions range: -0.026 to 0.892


In [14]:
from xgboost import XGBRegressor

# Try different simple settings
models = {
    'default': XGBRegressor(random_state=42),
    'more_trees': XGBRegressor(n_estimators=200, random_state=42),
    'deeper_trees': XGBRegressor(max_depth=6, random_state=42),
    'slower_learning': XGBRegressor(learning_rate=0.1, random_state=42)
}

for name, model in models.items():
    model.fit(X, y)
    predictions = model.predict(X_test)
    submission['accident_risk'] = predictions
    submission.to_csv(f'{name}_submission.csv', index=False)
    print(f"✅ {name} submission saved")

✅ default submission saved
✅ more_trees submission saved
✅ deeper_trees submission saved
✅ slower_learning submission saved


In [16]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
# Try different models
models = {
    'XGBoost': XGBRegressor(random_state=42),
    
    'LinearRegression': LinearRegression()
}

for name, model in models.items():
    model.fit(X, y)
    predictions = model.predict(X_test)
    submission['accident_risk'] = predictions
    submission.to_csv(f'{name}_submission.csv', index=False)
    print(f"✅ {name} predictions: {predictions.min():.3f} to {predictions.max():.3f}")

✅ XGBoost predictions: -0.026 to 0.892
✅ LinearRegression predictions: -0.077 to 0.828


In [31]:
from sklearn.ensemble import RandomForestRegressor


model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)


print("Training the Random Forest model...")
model.fit(X, y)
predictions = model.predict(X_test)
print(f"✅ {name} predictions: {predictions.min():.3f} to {predictions.max():.3f}")
print("Model training complete!")

Training the Random Forest model...
✅ LinearRegression predictions: 0.016 to 0.903
Model training complete!


In [18]:
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
predictions = model.predict(X)
mae = mean_absolute_error(y, predictions)



rmse= mean_squared_error(y, predictions, squared=False)

In [19]:
print(f"RMSE: {rmse}")
print(f"MAE: {mae}")

RMSE: 0.023189952097040376
MAE: 0.017657003021712258


In [22]:

print("\nFirst 10 predictions (formatted):")
for i, pred in enumerate(predictions[:10]):
    print(f"  {i+1}: {pred:.4f}")


First 10 predictions (formatted):
  1: 0.1287
  2: 0.3320
  3: 0.3277
  4: 0.1850
  5: 0.5299
  6: 0.6180
  7: 0.2430
  8: 0.1430
  9: 0.1902
  10: 0.1416


In [30]:
# If you want a mix, you need to manually combine them
model1 = XGBRegressor()


model1.fit(X, y)


print(f"Model type: {type(model1)}")
print("XGBoost MAE:", mean_absolute_error(y, model1.predict(X)))





Model type: <class 'sklearn.ensemble._forest.RandomForestRegressor'>
Model type: <class 'xgboost.sklearn.XGBRegressor'>
XGBoost MAE: 0.04316230386958962
